# 04 Federated Learning Review

This notebook reviews the shared global model from the federated-learning demo.


## Goals

- inspect the current global model
- review model round and feature order
- compare local and federated score fields in ingested events
- connect FL to privacy and governance


In [ ]:
from pathlib import Path
import json
import pandas as pd

candidate_paths = [
    Path("/workspace/federated_shared/global_model.json"),
    Path("/app/federated_shared/global_model.json"),
    Path("/workspace/data/global_model.json"),
]
existing = [p for p in candidate_paths if p.exists()]
print("Existing candidate paths:", existing)


In [ ]:
model = None
loaded_from = None
for p in existing:
    with open(p, "r", encoding="utf-8") as f:
        model = json.load(f)
    loaded_from = p
    break

print("Loaded from:", loaded_from)
model


In [ ]:
if model:
    display(pd.DataFrame({
        "feature": model.get("feature_order", []),
        "coefficient": model.get("coef", []),
    }))
    print("Intercept:", model.get("intercept"))
    print("Round:", model.get("round"))
    print("Clients:", model.get("num_clients"))
    print("Total samples:", model.get("total_samples"))
else:
    print("No global model found yet. Run the federated clients and aggregator first.")


In [ ]:
ingested_path = Path("/workspace/data/ingested_events.jsonl")
rows = []
if ingested_path.exists():
    with open(ingested_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

fed_rows = []
for r in rows:
    dr = r.get("detector_result", {})
    fr = dr.get("federated_result", {})
    fed_rows.append({
        "risk_score_rule": dr.get("risk_score_rule"),
        "risk_score_final": dr.get("risk_score_final"),
        "used_federated": fr.get("used_federated"),
        "risk_score_fl": fr.get("risk_score_fl"),
        "reason": fr.get("reason"),
        "action": dr.get("action"),
    })

pd.DataFrame(fed_rows).head()


## Reflection

In this lab, federated learning is a second-opinion mechanism:

- local data stay with campus clients
- only model updates are shared
- the final action remains bounded and policy-governed
